In [1]:
import os

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from great_tables import GT

conn = duckdb.connect()

load_dotenv(find_dotenv())


True

In [2]:
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD")
MINIO_ENDPOINT_DUCKDB = os.getenv("MINIO_ENDPOINT_DUCKDB")
# Install and load spatial extension
conn.execute("INSTALL spatial; LOAD spatial;")

# for S3/MinIO access
conn.execute("INSTALL httpfs")
conn.execute("LOAD httpfs")

#  MinIO connection
conn.execute("SET s3_region = 'us-east-1'")  # Region doesn't matter here
conn.execute(f"SET s3_endpoint = '{MINIO_ENDPOINT_DUCKDB}'")
conn.execute(f"SET s3_access_key_id = '{MINIO_ACCESS_KEY}'")
conn.execute(f"SET s3_secret_access_key = '{MINIO_SECRET_KEY}'")
conn.execute("SET s3_use_ssl = false")
conn.execute("SET s3_url_style = 'path'")  # Important for MinIO!

In [3]:
# Set up matplotlib style
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "#f0f0f0"
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10

In [4]:
result = conn.execute("""
    SELECT
    c.name,
    p.year,
    p.population
FROM read_parquet('s3://lakehouse-processed/dim_communes_france.parquet') AS c
JOIN read_parquet('s3://lakehouse-processed/fact_population.parquet') AS p
    ON c.id = p.id
WHERE c.id='09160'
ORDER BY year DESC
LIMIT 5
""").df()
print(result)

        name  year  population
0  Lavelanet  2023        5987
1  Lavelanet  2022        6018
2  Lavelanet  2021        6035
3  Lavelanet  2020        6052
4  Lavelanet  2019        6031


In [5]:
result = conn.execute("""
    SELECT
    *
FROM read_parquet('s3://lakehouse-processed/dim_communes_france.parquet') AS c
JOIN read_parquet('s3://lakehouse-processed/fact_population.parquet') AS p
    ON c.id = p.id
WHERE c.name='Bordeaux'
AND P.year >=2007
ORDER BY p.year DESC
""").df()
result

,id,current_code,name,name_upper,name_lower,siren_code,arrondissement_name,arrondissement_code,department_name,department_code,...,centroid,bbox_xmin,bbox_xmax,bbox_ymin,bbox_ymax,perimeter,number_enclaves,id_1,year,population
0,33063,33063,Bordeaux,BORDEAUX,bordeaux,213300635,Bordeaux,332,Gironde,33,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",-0.638699,-0.533325,44.810741,44.916694,0.464519,0,33063,2023,267991
1,33063,33063,Bordeaux,BORDEAUX,bordeaux,213300635,Bordeaux,332,Gironde,33,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",-0.638699,-0.533325,44.810741,44.916694,0.464519,0,33063,2022,265328
2,33063,33063,Bordeaux,BORDEAUX,bordeaux,213300635,Bordeaux,332,Gironde,33,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",-0.638699,-0.533325,44.810741,44.916694,0.464519,0,33063,2021,261804
3,33063,33063,Bordeaux,BORDEAUX,bordeaux,213300635,Bordeaux,332,Gironde,33,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",-0.638699,-0.533325,44.810741,44.916694,0.464519,0,33063,2020,259809
4,33063,33063,Bordeaux,BORDEAUX,bordeaux,213300635,Bordeaux,332,Gironde,33,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",-0.638699,-0.533325,44.810741,44.916694,0.464519,0,33063,2019,260958
5,33063,33063,Bordeaux,BORDEAUX,bordeaux,213300635,Bordeaux,332,Gironde,33,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",-0.638699,-0.533325,44.810741,44.916694,0.464519,0,33063,2018,257068
6,33063,33063,Bordeaux,BORDEAUX,bordeaux,213300635,Bordeaux,332,Gironde,33,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",-0.638699,-0.533325,44.810741,44.916694,0.464519,0,33063,2017,254436
7,33063,33063,Bordeaux,BORDEAUX,bordeaux,213300635,Bordeaux,332,Gironde,33,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",-0.638699,-0.533325,44.810741,44.916694,0.464519,0,33063,2016,252040
8,33063,33063,Bordeaux,BORDEAUX,bordeaux,213300635,Bordeaux,332,Gironde,33,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",-0.638699,-0.533325,44.810741,44.916694,0.464519,0,33063,2015,249712
9,33063,33063,Bordeaux,BORDEAUX,bordeaux,213300635,Bordeaux,332,Gironde,33,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",-0.638699,-0.533325,44.810741,44.916694,0.464519,0,33063,2014,246586


In [13]:
result = conn.execute("""
    SELECT AVG(CASE WHEN c.name = 'Lavelanet' THEN p.mean_salary END) AS mean_lavelanet,
    AVG(CASE WHEN c.department_name = 'Ariège' THEN p.mean_salary END) AS mean_ariege,
    AVG(CASE WHEN c.region_name = 'Occitanie' THEN p.mean_salary END) AS mean_occitanie
    FROM read_parquet('s3://lakehouse-processed/dim_communes_france.parquet') c
        JOIN read_parquet('s3://lakehouse-processed/fact_salaries.parquet') p ON c.id = p.id
""").df()
result

,mean_lavelanet,mean_ariege,mean_occitanie
0,1898.5,1989.857143,2037.990546


In [7]:
stop

NameError: name 'stop' is not defined

In [ ]:
def get_commune_population(commune_id: str, conn) -> pd.DataFrame:
    """
    Query population data for a specific commune by ID.

    Args:
        commune_id: The commune identifier (string)
        conn: Optional DuckDB connection. If None, creates a new one.

    Returns:
        DataFrame with name, year, population columns sorted by year descending
    """

    query = """
    SELECT
        c.name,
        c.department_name,
        c.department_code,
        p.year,
        p.population,
        p2.population AS previous_population,
        p.population - previous_population AS 'Population Evolution',
        ROUND((p.population - p2.population) * 100.0 / p2.population, 2) AS "Population Evolution (%)"
    FROM read_parquet('s3://lakehouse-processed/dim_communes_france.parquet') AS c
    JOIN read_parquet('s3://lakehouse-processed/fact_population.parquet') AS p
        ON c.id = p.id
    LEFT JOIN  read_parquet('s3://lakehouse-processed/fact_population.parquet') AS p2
        ON c.id = p2.id
        AND p.id = p2.id
        AND p.year-1=p2.year
    WHERE c.id=?
    AND P.year >=2007
    ORDER BY p.year DESC
    """

    df = conn.execute(query, [commune_id]).df()

    if df.empty:
        print(f"Warning: No data found for commune ID: {commune_id}")

    return df

In [ ]:
# Generate report for Lavelanet
commune_id = "33063"

In [ ]:
df = get_commune_population(commune_id, conn)
df.head()


In [ ]:
def create_population_table(df: pd.DataFrame, title: str = None) -> GT:
    if df.empty:
        return None

    commune_name = df["name"].iloc[0]
    department_name = df["department_name"].iloc[0]
    department_code = df["department_code"].iloc[0]
    max_year = df["year"].iloc[0]
    min_year = df["year"].iloc[-1]

    table_df = df[
        [
            "year",
            "population",
            "Population Evolution",
            "Population Evolution (%)",
        ]
    ].copy()

    # Add colored arrows to Population Evolution
    def _fmt_arrow(v, fmt: str, suffix: str = "") -> str:
        if pd.isna(v):
            return "—"
        if v > 0:
            return f'<span style="color:green">&#9650; {v:{fmt}}{suffix}</span>'
        if v < 0:
            return f'<span style="color:red">&#9660; {v:{fmt}}{suffix}</span>'
        return f'<span style="color:gray">&#9644; {v:{fmt}}{suffix}</span>'

    table_df["Population Evolution"] = table_df["Population Evolution"].apply(
        lambda v: _fmt_arrow(v, "+,.0f")
    )
    table_df["Population Evolution (%)"] = table_df["Population Evolution (%)"].apply(
        lambda v: _fmt_arrow(v, "+.1f", suffix="%")
    )

    return (
        GT(table_df)
        .tab_header(
            title=f"Population for {commune_name} ({department_name} - {department_code})",
            subtitle=f"Data for Years {min_year} - {max_year}",
        )
        .fmt_integer(columns="population")
        .fmt_markdown(columns="Population Evolution")
        .fmt_markdown(columns="Population Evolution (%)")
        .cols_align(
            align="right", columns=["Population Evolution", "Population Evolution (%)"]
        )
    )

In [ ]:
gt_table = create_population_table(df)
gt_table.save(f"{commune_id}.png")
gt_table.show()